# 07 · Inference, bulk translation and export

**What this notebook is for**

1. A simple `translate()` you can call on any sentence.
2. Bulk-translating the English PSA corpus into Ekegusii — which produces the
   candidate data for human post-editing, the highest-value next step.
3. Saving and optionally publishing the model.

**Runtime** — minutes for spot checks; 1–2 hours for the full 50k corpus.

In [18]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
sys.path.insert(0, str(pathlib.Path.cwd()))
import nb_common as C

C.set_seed()
C.use_house_style()
print(f"project root: {C.ROOT}")

project root: /home/jovyan/public-service-anouncement-MT


In [19]:
import torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(C.FINETUNED_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(
    C.FINETUNED_MODEL, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device).eval()
print(f"loaded {C.FINETUNED_MODEL.name} on {device}")

Loading weights: 100%|██████████| 509/509 [00:00<00:00, 1212.42it/s]


loaded nllb600m-stage2-psa on cuda


## 1. Translate anything

In [20]:
EOS = tok.eos_token_id

@torch.no_grad()
def translate(texts, src_lang=C.ENG, tgt_lang=C.GUZ, batch=32, beams=4, max_len=128):
    if isinstance(texts, str):
        texts = [texts]
    out = []
    tgt_id = tok.convert_tokens_to_ids(tgt_lang)
    pad = tok.pad_token_id
    for i in range(0, len(texts), batch):
        chunk = texts[i:i + batch]
        enc = [[tok.convert_tokens_to_ids(src_lang)] +
               tok(t, add_special_tokens=False, truncation=True,
                   max_length=max_len - 2)["input_ids"] + [EOS] for t in chunk]
        m = max(len(e) for e in enc)
        ids = torch.tensor([[pad] * (m - len(e)) + e for e in enc]).to(device)
        gen = model.generate(input_ids=ids, attention_mask=(ids != pad).long(),
                             forced_bos_token_id=tgt_id,
                             max_new_tokens=max_len, num_beams=beams)
        out += tok.batch_decode(gen, skip_special_tokens=True)
    return out

samples = [
    "Report suspected cholera cases to the nearest health facility immediately.",
    "Apply for HELB loans before the September deadline through the online portal.",
    "Vaccinate your livestock against foot and mouth disease before the rains begin.",
    "Avoid crossing flooded rivers and follow county safety guidance.",
]
for lang, name in [(C.GUZ, "Ekegusii"), (C.SWH, "Kiswahili")]:
    print(f"\n--- English -> {name} ---")
    for s, t in zip(samples, translate(samples, C.ENG, lang)):
        print(f"  EN : {s}")
        print(f"  {name[:3].upper()}: {t}\n")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- English -> Ekegusii ---


[transformers] Both `max_new_tokens` (=128) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  EN : Report suspected cholera cases to the nearest health facility immediately.
  EKE: Karwe amang'ana igoro ya baria bagosemigwa ng'a mbare noborwaire bwa cholera ase enyomba y'obobwenia ere ang'e nainwe bwango.

  EN : Apply for HELB loans before the September deadline through the online portal.
  EKE: Kora ogosaba kwao ase HELB nyuma yomotienyi 9 goetera omoroberio bweintaneti.

  EN : Vaccinate your livestock against foot and mouth disease before the rains begin.
  EKE: Renta chingiti chiao korwa ase oborwaire bw'amagoro na obw'omonwa chingaki embura etaracha.

  EN : Avoid crossing flooded rivers and follow county safety guidance.
  EKE: Kabe maiso gotamboka chindooche chire ne chindooche na gotegerera oborendi bwa'kaunti.


--- English -> Kiswahili ---
  EN : Report suspected cholera cases to the nearest health facility immediately.
  KIS: Karwe ripoti ya haraka igoro ya baria bagosemigwa korwara cholera gochia chinyomba chia afya chiao chire ang'e nao.

  EN : Apply for HELB l

## 2. Bulk-translate the PSA corpus into Ekegusii

This is not a finished dataset — it is **machine output awaiting human
post-editing**. Post-editing a few thousand of these with native speakers gives
you genuine Ekegusii PSA data, which is worth far more than any amount of extra
synthetic text and is the natural second iteration of this project.

In [21]:
RUN_BULK = False          # set True when you are ready to spend the GPU time
LIMIT    = None           # e.g. 5000 to do a slice first

if RUN_BULK:
    psa = pd.read_csv(C.PSA_PARALLEL_CSV)
    if LIMIT:
        psa = psa.head(LIMIT)
    texts = psa["English"].astype(str).tolist()
    print(f"translating {len(texts):,} PSAs -> Ekegusii ...")
    psa["Ekegusii_mt"] = translate(texts, C.ENG, C.GUZ, batch=64)
    psa["needs_post_editing"] = True
    out = C.OUTPUT / "psa_ekegusii_machine_draft.csv"
    psa.to_csv(out, index=False, encoding="utf-8")
    print(f"wrote {out}")
else:
    print("RUN_BULK is False - set it to True to translate the whole corpus.")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


translating 50,317 PSAs -> Ekegusii ...


[transformers] Both `max_new_tokens` (=128) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

wrote /home/jovyan/public-service-anouncement-MT/output/psa_ekegusii_machine_draft.csv


## 3. Post-editing worksheet

Give your native speakers something easy to work with: a stratified sample with
an empty column to correct in, and a rating column so you also learn *how* wrong
the model is.

In [22]:
SAMPLE_PER_DOMAIN = 60

draft_path = C.OUTPUT / "psa_ekegusii_machine_draft.csv"
if draft_path.exists():
    d = pd.read_csv(draft_path)
    parts = [g.sample(min(SAMPLE_PER_DOMAIN, len(g)), random_state=C.SEED)
             for _, g in d.groupby("Domain")] if "Domain" in d.columns else [d.head(300)]
    sheet = pd.concat(parts)[["PSA_Id", "Domain", "English", "Ekegusii_mt"]] \
        if "PSA_Id" in d.columns else pd.concat(parts)[["English", "Ekegusii_mt"]]
    sheet["Ekegusii_corrected"] = ""
    sheet["adequacy_1_to_4"] = ""
    sheet["register_ok_y_n"] = ""
    sheet["notes"] = ""
    out = C.OUTPUT / "post_editing_worksheet.csv"
    sheet.to_csv(out, index=False, encoding="utf-8")
    print(f"wrote {len(sheet)} rows -> {out}")
    print("\nadequacy: 1 = meaning lost, 4 = fully correct")
    print("register: does it sound like a public notice rather than scripture?")
else:
    print("Run section 2 first to produce the machine draft.")

wrote 300 rows -> /home/jovyan/public-service-anouncement-MT/output/post_editing_worksheet.csv

adequacy: 1 = meaning lost, 4 = fully correct
register: does it sound like a public notice rather than scripture?


## 4. Export and publish all three systems

The paper compares four systems and three of them are checkpoints produced here.
Publishing only stage 2 would leave the **baseline** every reported gain is
measured against, and the **control** that justifies the curriculum, unavailable
— which makes the headline number unverifiable by anyone else.

GitHub cannot hold them: 100 MB per file, hard, against ~2.4 GB of weights each.

Three things this cell refuses to get wrong:

- **The tokenizer is pushed with every model.** `guz_Latn` is an *added* token.
  Weights without that tokenizer are weights whose Ekegusii is unreachable.
- **`private=True` only takes effect when the repository is created.** If the
  repo already exists as public, the argument is silently ignored and you must
  change visibility in the repository's Settings.
- **Each repo gets a card.** Stage 2 gets `MODEL_CARD.md`; the other two get a
  short card naming their role and pointing at it, so nobody downloads the
  control by mistake and reports its numbers as the result.

In [ ]:
import gc, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

PUSH_TO_HUB = False
HF_USER     = ""        # <- your Hugging Face username, e.g. "samuelabrha"
HF_PRIVATE  = True      # read the licensing note at the bottom before changing

REPOS = {
    "stage1": (C.STAGE1_MODEL, "nllb-200-600M-ekegusii-stage1",
               "Stage 1 - general Ekegusii, fine-tuned on the Bible and "
               "storybook corpus. The baseline every PSA gain is measured against."),
    "stage2": (C.STAGE2_MODEL, "nllb-200-600M-ekegusii-psa",
               "Two-stage curriculum - kept as the ablation that lost. Published "
               "so the negative result can be checked, NOT for use."),
    "mixed":  (C.MIXED_MODEL,  "nllb-200-600M-ekegusii-mixed",
               "THE RELEASED MODEL. All data in a single pass. Best on every "
               "test set, including scripture."),
}

if PUSH_TO_HUB:
    assert HF_USER and HF_USER != "your-username", \
        "Set HF_USER to your actual Hugging Face username first"
    from huggingface_hub import notebook_login, upload_file, whoami
    try:
        whoami()                      # already authenticated?
    except Exception:
        notebook_login()

for name, (path, repo_name, blurb) in REPOS.items():
    if not path.exists():
        print(f"  --  {name:<7} {path.name} is not on this machine - skipped")
        continue

    model_ = AutoModelForSeq2SeqLM.from_pretrained(path)
    tok_ = AutoTokenizer.from_pretrained(path)

    # The added language token IS the contribution. A checkpoint without it is
    # not worth uploading, so fail here rather than publish a dud.
    guz_id = tok_.convert_tokens_to_ids(C.GUZ)
    assert guz_id != tok_.unk_token_id, f"{path} has no {C.GUZ} token - do not publish"

    export = C.ARTIFACTS / "export" / path.name
    export.mkdir(parents=True, exist_ok=True)
    model_.save_pretrained(export)
    tok_.save_pretrained(export)
    print(f"  ok  {name:<7} {C.GUZ} id {guz_id}, vocab {len(tok_):,}  ->  {export}")

    if PUSH_TO_HUB:
        repo = f"{HF_USER}/{repo_name}"
        model_.push_to_hub(repo, private=HF_PRIVATE)
        # Never skip this line. Without the tokenizer the added token is gone.
        tok_.push_to_hub(repo, private=HF_PRIVATE)

        card = C.ROOT / "MODEL_CARD.md"
        # MODEL_CARD.md describes the RELEASED model, which is the single-pass
        # one. Stage 2 lost the ablation; giving it the full card would invite
        # someone to download it and report its numbers as the result.
        if name == "mixed" and card.exists():
            upload_file(path_or_fileobj=str(card), path_in_repo="README.md",
                        repo_id=repo, repo_type="model")
        else:
            stub = (f"# {repo_name}\n\n{blurb}\n\n"
                    f"One of three checkpoints from a study on fine-tuning NLLB-200 "
                    f"for Ekegusii public service announcements. Method, data and the "
                    f"full results table live on the released model card:\n"
                    f"https://huggingface.co/{HF_USER}/nllb-200-600M-ekegusii-mixed\n")
            upload_file(path_or_fileobj=stub.encode("utf-8"),
                        path_in_repo="README.md", repo_id=repo, repo_type="model")
        print(f"      pushed -> https://huggingface.co/{repo}"
              f"  ({'private' if HF_PRIVATE else 'PUBLIC'})")

    del model_, tok_
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if not PUSH_TO_HUB:
    print("\nPUSH_TO_HUB is False - exported locally only. "
          "Set it, and HF_USER, to publish.")

# NOTE ON LICENSING: the weights are a derivative of the Ekegusii Revised Bible
# (c) Bible Society of Kenya and of PSA data whose provenance the supervisor has
# not specified. Publish PRIVATE first and confirm what may be released.

## 5. Verify the uploads round-trip

An upload that "succeeded" and a model someone else can actually use are not the
same thing. This cell throws away everything local and pulls each repository
back from the Hub by name, exactly as a stranger would — then checks the added
token survived and that the model still emits Ekegusii.

It downloads ~7 GB. Run it once, after publishing, and never think about it
again. `tests/verify_hf_uploads.py` does the same from a terminal.

In [ ]:
# Set to True after pushing. Downloads ~7 GB.
VERIFY_HUB = False

if VERIFY_HUB:
    assert HF_USER, "Set HF_USER first"
    import torch
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    PROBE = "Report suspected cholera cases to the nearest health facility."
    ok_all = True

    for name, (_, repo_name, _) in REPOS.items():
        repo = f"{HF_USER}/{repo_name}"
        try:
            t = AutoTokenizer.from_pretrained(repo)
            m = AutoModelForSeq2SeqLM.from_pretrained(repo).eval()
            guz_id = t.convert_tokens_to_ids(C.GUZ)
            assert guz_id != t.unk_token_id, f"{repo} downloaded WITHOUT {C.GUZ}"

            ids = ([t.convert_tokens_to_ids(C.ENG)]
                   + t(PROBE, add_special_tokens=False)["input_ids"]
                   + [t.eos_token_id])
            with torch.no_grad():
                out = m.generate(input_ids=torch.tensor([ids]),
                                 forced_bos_token_id=guz_id,
                                 max_new_tokens=64, num_beams=4)
            guz = t.batch_decode(out, skip_special_tokens=True)[0]
            assert guz.strip(), f"{repo} generated nothing"
            print(f"  ok  {name:<7} {repo}")
            print(f"      {C.GUZ} id {guz_id} | vocab {len(t):,}")
            print(f"      {guz}\n")
            del m, t
        except Exception as exc:
            ok_all = False
            print(f"  FAIL {name:<7} {repo}\n      {type(exc).__name__}: {exc}\n")

    print("all three recoverable from the Hub" if ok_all
          else "AT LEAST ONE REPOSITORY IS NOT USABLE - fix before citing it")
else:
    print("VERIFY_HUB is False - set it to True after pushing.")

## Where to go next

1. **Get the human PSA test set built** (100–300 sentences, stratified across the
   five domains). Until it exists, notebook 04 can only report biblical-domain
   Ekegusii quality, and no claim about PSA translation is supportable.
2. **Post-edit a few thousand machine drafts** from section 2 and retrain. This
   is the fastest route to genuine in-domain Ekegusii data.
3. **Chase civic vocabulary.** Notebook 01 showed that 53.6% of PSA content-word
   types never appear in the training data — institutions, portals, bursaries.
   County government notices, health leaflets and radio scripts in Ekegusii
   would close that gap in a way no amount of scripture can.